# 05 — Escoragem | Bank Marketing

Escora uma base no formato original (`bank-additional-full.csv`) **sem treinar nada**: todo o tratamento usa os parâmetros salvos do treino.

| Etapa | De onde vem |
|---|---|
| Variáveis novas (flags, faixas, `log_campaign`...) | Mesmas regras do notebook 02 |
| Imputação (medianas e modas) | `transformadores_prep.joblib` |
| Target Encoding e padronização | `transformadores_prep.joblib` |
| Lista de variáveis | `variaveis_selecionadas.json` (conferida com a lista salva no modelo) |
| Modelo | `lgbm_tunado.joblib` |
| Faixas de propensão | Tetos definidos no treino (escala `score_1000`) |

> **Regra:** nada aqui usa `fit`. Só `transform` e `predict_proba`.

## 1. Setup e caminhos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import roc_auc_score, roc_curve

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PASTA = r'/content/drive/MyDrive/PoD_Academy/Modelagem/Problema Propensao Compra Produto Bancario'
ARQ_BASE          = f'{PASTA}/Base/bank-additional-full.csv'
ARQ_TRANSF        = f'{PASTA}/Base/prep/transformadores_prep.joblib'
ARQ_VARIAVEIS     = f'{PASTA}/Base/FeatureSelection/variaveis_selecionadas.json'
ARQ_MODELO        = f'{PASTA}/Modelos/lgbm_tunado.joblib'
PASTA_SAIDA       = f'{PASTA}/Faixas de Risco'

os.makedirs(PASTA_SAIDA, exist_ok=True)

TETO_BAIXA = 57
TETO_MEDIA = 337

## 2. Leitura da base e dos artefatos

In [ ]:
df = pd.read_csv(ARQ_BASE, sep=';')
print(f'Base: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

transf = joblib.load(ARQ_TRANSF)
pacote_modelo = joblib.load(ARQ_MODELO)
modelo = pacote_modelo['modelo']

with open(ARQ_VARIAVEIS) as f:
    FEATURES = json.load(f)

# A ordem das colunas precisa ser exatamente a do treino do modelo
if 'features' in pacote_modelo and list(pacote_modelo['features']) != list(FEATURES):
    print('Atenção: a lista do JSON difere da salva no modelo. Usando a lista do modelo.')
    FEATURES = list(pacote_modelo['features'])

COLS_CAT, COLS_NUM = transf['COLS_CAT'], transf['COLS_NUM']
COLS_TE, COLS_ESCALAR = transf['COLS_TE'], transf['COLS_ESCALAR']
print(f'{len(FEATURES)} variáveis no modelo:', FEATURES)

## 3. Variáveis novas (mesmas regras do notebook 02)

A base é escorada inteira, linha a linha. Não removemos duplicatas aqui: numa escoragem, cada cliente recebe o seu score.

In [ ]:
base = df.copy()

# Safra (a base está em ordem cronológica)
MAPA_MES = {m: i for i, m in enumerate(
    ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec'], start=1)}
mes_num = base['month'].map(MAPA_MES)
base['AnoMes'] = (2008 + (mes_num.diff() < 0).cumsum()) * 100 + mes_num

# Target (só para conferência; a escoragem não depende dele)
if 'y' in base.columns:
    base['target'] = (base['y'] == 'yes').astype(int)

base['flag_nunca_contatado'] = (base['pdays'] == 999).astype(int)
base['pdays_tratado'] = base['pdays'].replace(999, np.nan)
base['flag_sucesso_anterior'] = (base['poutcome'] == 'success').astype(int)
base['flag_celular'] = (base['contact'] == 'cellular').astype(int)
base['contatos_totais'] = base['campaign'] + base['previous']
base['qtd_produtos'] = (base['housing'] == 'yes').astype(int) + (base['loan'] == 'yes').astype(int)
base['trimestre'] = 'T' + ((mes_num - 1) // 3 + 1).astype(str)
base['faixa_idade'] = pd.cut(base['age'], bins=[0, 24, 34, 44, 54, 64, 120],
                             labels=['17-24', '25-34', '35-44', '45-54', '55-64', '65+']).astype(str)
base['log_campaign'] = np.log1p(base['campaign'])

## 4. Imputação, Target Encoding e padronização (parâmetros do treino)

In [ ]:
# 4.1 Numéricas -> medianas do treino
base[COLS_NUM] = base[COLS_NUM].fillna(transf['medianas'])

# 4.2 Categóricas 'unknown' -> modas do treino (as colunas que ficaram fora do dicionário mantêm 'unknown')
for c, moda in transf['modas'].items():
    base[c] = base[c].replace('unknown', moda)

# 4.3 Target Encoding com o encoder ajustado no treino
base[COLS_TE] = transf['target_encoder'].transform(base[COLS_CAT])

# 4.4 Padronização com o scaler ajustado no treino (nas mesmas colunas em que ele foi ajustado)
X_escalado = base[COLS_ESCALAR].copy()
X_escalado[COLS_ESCALAR] = transf['scaler'].transform(X_escalado[COLS_ESCALAR])
for c in COLS_ESCALAR:
    base[c] = X_escalado[c]

X = base[FEATURES]
assert X.isna().sum().sum() == 0, 'Há nulos nas variáveis do modelo'
print('Variáveis prontas:', X.shape)

## 5. Escoragem e faixas de propensão

In [ ]:
base['score'] = modelo.predict_proba(X)[:, 1]
base['score_1000'] = np.round(base['score'] * 1000).astype(int)

def faixa_propensao(b):
    condicoes = [b['score_1000'] > TETO_MEDIA,
                 (b['score_1000'] > TETO_BAIXA) & (b['score_1000'] <= TETO_MEDIA),
                 b['score_1000'] <= TETO_BAIXA]
    rotulos = ['1) Alta propensao', '2) Media propensao', '3) Baixa propensao']
    return np.select(condicoes, rotulos, default='Sem faixa')

base['faixa_propensao'] = faixa_propensao(base)
assert (base['faixa_propensao'] != 'Sem faixa').all()

resumo = base.groupby('faixa_propensao').agg(
    qtd=('score', 'size'), score_1000_min=('score_1000', 'min'), score_1000_max=('score_1000', 'max'))
resumo['pct_publico'] = resumo['qtd'] / resumo['qtd'].sum()
if 'target' in base.columns:
    resumo['taxa_conversao'] = base.groupby('faixa_propensao')['target'].mean()
resumo.style.format({'qtd': '{:,.0f}', 'pct_publico': '{:.1%}', 'taxa_conversao': '{:.2%}'})

## 6. Salvar a base escorada

In [ ]:
cols_saida = list(df.columns) + ['AnoMes', 'score', 'score_1000', 'faixa_propensao']
cols_saida = [c for c in dict.fromkeys(cols_saida)]          # sem repetição, na ordem
base_escorada = df.assign(**{c: base[c] for c in ['AnoMes', 'score', 'score_1000', 'faixa_propensao']})[cols_saida]

base_escorada.to_parquet(f'{PASTA_SAIDA}/base_completa_escorada.parquet', index=False)
print('Salvo em', PASTA_SAIDA)
base_escorada.head()

A base salva mantém **as colunas originais** (sem encoding nem padronização) e acrescenta `AnoMes`, `score`, `score_1000` e `faixa_propensao`, para ficar fácil de ler e de levar para o Excel.